# The Markov blanket

_Artificial Intelligence — more_

**A node only needs to know its neighbors. Given them, the rest of the world is irrelevant.**

In a Bayes net, a node is not tangled up with the whole graph. It depends on a small local set: its Markov blanket.

     The blanket is the node's parents, its children, and its children's other parents (co-parents).

---

This notebook builds the Markov blanket from scratch, one step at a time. Run each cell, read the note above it, and see how a node's local neighborhood is enough to reason about it. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

We'll compute the Markov blanket of a node in a small Bayes net. We build it in two steps: (1) the graph plus a helper to find a node's children, (2) assembling the blanket from parents, children, and co-parents.

### Step 1 — Describe the graph and find a node's children

We store the net as a dict mapping each node to its list of parents. Children aren't stored directly, so the `children` helper scans every node and keeps those that list our target as a parent. Here `X` has parents `A` and `B`, and is a parent of `Y` and `Z`.

In [ ]:
# Bayes net edges (parent -> child). Target node = "X".
parents = {"X": ["A", "B"], "Y": ["X", "C"], "Z": ["X", "D"], "W": ["E"]}

def children(node):
    # A node's children are every node that lists it among its parents.
    return [n for n, ps in parents.items() if node in ps]

### Step 2 — Assemble the blanket: parents, children, and co-parents

The Markov blanket of a node is its parents, its children, and its children's *other* parents (the co-parents). We start from the parents, then for each child add the child itself and all of that child's parents. We drop the node itself at the end. `E` and `W` lie outside the blanket, so `X` is independent of them given the blanket.

In [ ]:
def markov_blanket(node):
    blanket = set(parents.get(node, []))      # start with the parents
    for ch in children(node):                 # add each child ...
        blanket.add(ch)
        blanket.update(parents[ch])           # ... and that child's other parents (co-parents)
    blanket.discard(node)                     # a node is never in its own blanket
    return sorted(blanket)

mb = markov_blanket("X")
print("children of X:", children("X"))
print("Markov blanket of X:", mb)

# E and W are outside the blanket -> X is independent of them given the blanket.
print("W in blanket?", "W" in mb)

## Visualize the data & results

_Burglar Alarm network: given an earthquake and both neighbors calling, what is the local posterior that the alarm is on?_

### Step 1 — Define the Burglar Alarm network

The Alarm has two parents, Burglary and Earthquake, and two children, JohnCalls and MaryCalls — exactly the Alarm's Markov blanket. Each table is a conditional probability: `P_A_BE[b, e]` gives `P(Alarm | Burglary=b, Earthquake=e)`, while `P_J_A` and `P_M_A` give each neighbor's chance of calling given the alarm state.

In [ ]:
# Burglar Alarm net. Markov blanket of Alarm = parents {Burglary, Earthquake} + children {John, Mary}.
P_A_BE = np.zeros((2, 2, 2))                       # P(Alarm | Burglary, Earthquake)
P_A_BE[0, 0] = [0.999, 0.001]
P_A_BE[0, 1] = [0.71, 0.29]
P_A_BE[1, 0] = [0.06, 0.94]
P_A_BE[1, 1] = [0.05, 0.95]

P_J_A = np.array([[0.95, 0.05], [0.10, 0.90]])    # P(JohnCalls | Alarm)
P_M_A = np.array([[0.99, 0.01], [0.30, 0.70]])    # P(MaryCalls | Alarm)

### Step 2 — Compute the local posterior over the Alarm

We observe the full blanket: no burglary, an earthquake, and both John and Mary calling. The posterior over the Alarm needs only these blanket factors — the parents' table times each child's calling likelihood — which we score for Alarm off and on, then normalize. Conditioning on the blanket alone is enough.

In [ ]:
# Observed: no burglary, an earthquake, John & Mary both called.
B = 0
E = 1
J = 1
M = 1

# Score each Alarm state using only the blanket factors, then normalize.
score = np.array([P_A_BE[B, E, a] * P_J_A[a, J] * P_M_A[a, M] for a in range(2)])
post = score / score.sum()        # [P(off), P(on)] -> on approx 0.998

### Step 3 — Plot the local posterior

The two bars show `P(Alarm = On)` and `P(Alarm = Off)` given the blanket. The On bar dominates: an earthquake plus two calls makes the alarm almost certainly on, even with no burglary — and we never needed any node outside the blanket to reach that conclusion.

In [ ]:
labels = ["Alarm = On", "Alarm = Off"]
vals = [post[1], post[0]]

fig, ax = plt.subplots()
bars = ax.bar(labels, vals, color=["#7ee787", "#ff7b72"])
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v, str(round(v, 3)), ha="center", va="bottom")
ax.set_title("P(Alarm | its Markov blanket) in the Burglar Alarm net")
plt.show()